# 01 — Data Understanding

**Business question:** Which e-commerce customers are likely to churn, why, and how can nature-inspired algorithms (GA / PSO) improve prediction?

This notebook frames the data-mining problem, documents each feature, checks data quality, and saves a clean interim snapshot. No modelling yet.

In [1]:
from pathlib import Path
import random

import numpy as np

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

cwd = Path.cwd().resolve()
PROJECT_ROOT = next(
    (p for p in [cwd, *cwd.parents] if (p / "environment.yml").exists()),
    cwd,
)

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_INTERIM = PROJECT_ROOT / "data" / "interim"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
FIGURES_DIR = PROJECT_ROOT / "reports" / "figures"
REPORTS_DIR = PROJECT_ROOT / "reports"
MODELS_DIR = PROJECT_ROOT / "models"

for d in (DATA_INTERIM, DATA_PROCESSED, FIGURES_DIR, MODELS_DIR):
    d.mkdir(parents=True, exist_ok=True)

print(f"Project root : {PROJECT_ROOT}")
print(f"Random seed  : {RANDOM_SEED}")
print(f"Raw data     : {DATA_RAW}")
print(f"Interim      : {DATA_INTERIM}")
print(f"Processed    : {DATA_PROCESSED}")
print(f"Figures      : {FIGURES_DIR}")
print(f"Models       : {MODELS_DIR}")

Project root : C:\Users\ishan\OneDrive\Documents\Work\3 Year 2 Semester\IT41033 - Nature Inspired Algorithms\Mini Project\ecommerce-customer-churn-prediction
Random seed  : 42
Raw data     : C:\Users\ishan\OneDrive\Documents\Work\3 Year 2 Semester\IT41033 - Nature Inspired Algorithms\Mini Project\ecommerce-customer-churn-prediction\data\raw
Interim      : C:\Users\ishan\OneDrive\Documents\Work\3 Year 2 Semester\IT41033 - Nature Inspired Algorithms\Mini Project\ecommerce-customer-churn-prediction\data\interim
Processed    : C:\Users\ishan\OneDrive\Documents\Work\3 Year 2 Semester\IT41033 - Nature Inspired Algorithms\Mini Project\ecommerce-customer-churn-prediction\data\processed
Figures      : C:\Users\ishan\OneDrive\Documents\Work\3 Year 2 Semester\IT41033 - Nature Inspired Algorithms\Mini Project\ecommerce-customer-churn-prediction\reports\figures
Models       : C:\Users\ishan\OneDrive\Documents\Work\3 Year 2 Semester\IT41033 - Nature Inspired Algorithms\Mini Project\ecommerce-custome

## 1. Load raw Excel

The Kaggle workbook has two sheets: `Data Dict` (definitions) and `E Comm` (customer rows).

In [ ]:
import pandas as pd

RAW_XLSX = DATA_RAW / "E_Commerce_Dataset.xlsx"
assert RAW_XLSX.exists(), f"Missing dataset: {RAW_XLSX}"

data_dict_raw = pd.read_excel(RAW_XLSX, sheet_name="Data Dict", header=None)
df = pd.read_excel(RAW_XLSX, sheet_name="E Comm")

print("E Comm shape:", df.shape)
print("Columns:", list(df.columns))
df.head()

## 2. Feature dictionary

We rebuild the dictionary from the Data Dict sheet and add short business notes that matter for churn analysis.

In [ ]:
# Parse Data Dict sheet (header row is embedded)
dd = data_dict_raw.copy()
# Find the header-like row
header_idx = dd.index[dd.apply(lambda r: r.astype(str).str.contains("Variable", na=False).any(), axis=1)][0]
dd2 = dd.iloc[header_idx + 1 :, [1, 2, 3]].copy()
dd2.columns = ["Table", "Variable", "Description"]
dd2 = dd2.dropna(subset=["Variable"]).reset_index(drop=True)

business_notes = {
    "CustomerID": "Surrogate key — drop from models; use only for joins / examples.",
    "Churn": "Target (1 = churned, 0 = retained). Class is imbalanced.",
    "Tenure": "Relationship length. Short tenure often correlates with higher churn risk.",
    "PreferredLoginDevice": "Channel preference (Phone / Mobile Phone / Computer). May need harmonisation.",
    "CityTier": "Urbanisation tier (ordinal 1–3). Proxy for market maturity / logistics.",
    "WarehouseToHome": "Delivery distance. Longer distance can hurt experience.",
    "PreferredPaymentMode": "Payment preference. Overlapping labels (CC vs Credit Card, COD vs Cash on Delivery).",
    "Gender": "Demographic attribute.",
    "HourSpendOnApp": "Engagement proxy. Missingness is common.",
    "NumberOfDeviceRegistered": "Multi-device usage — engagement / account complexity.",
    "PreferedOrderCat": "Category affinity (note spelling Prefered). Mobile vs Mobile Phone overlap.",
    "SatisfactionScore": "Ordinal service score (1–5). Key retention signal.",
    "MaritalStatus": "Demographic segment.",
    "NumberOfAddress": "Address book size — possible multi-location / switching signal.",
    "Complain": "Binary complaint flag last month — strong churn risk signal.",
    "OrderAmountHikeFromlastYear": "Spend growth. Declining hike may signal disengagement.",
    "CouponUsed": "Promo sensitivity.",
    "OrderCount": "Recent purchase frequency.",
    "DaySinceLastOrder": "Recency. High values = dormant risk (but watch leakage vs churn definition).",
    "CashbackAmount": "Reward intensity. Low cashback + complaints may amplify churn.",
}

feat_dict = dd2.copy()
feat_dict["BusinessNote"] = feat_dict["Variable"].map(business_notes)
feat_dict.to_csv(DATA_INTERIM / "feature_dictionary.csv", index=False)
feat_dict

## 3. Schema, missingness, duplicates

In [ ]:
print("Dtypes:")
print(df.dtypes)
print("\nMissing counts:")
missing = df.isna().sum()
print(missing[missing > 0])
print(f"\nRows with any NA: {df.isna().any(axis=1).sum()} ({df.isna().any(axis=1).mean():.1%})")
print(f"CustomerID unique: {df['CustomerID'].nunique()} / {len(df)}")
print(f"Duplicate CustomerID rows: {df['CustomerID'].duplicated().sum()}")
print(f"Fully duplicate rows: {df.duplicated().sum()}")

In [ ]:
import matplotlib.pyplot as plt

# Visual: which columns are incomplete?
miss_pct = (df.isna().mean() * 100).loc[lambda s: s > 0].sort_values(ascending=True)
fig, ax = plt.subplots(figsize=(7, 4))
ax.barh(miss_pct.index.astype(str), miss_pct.values, color="#e9c46a")
ax.set_xlabel("Missing %")
ax.set_title("Missing values by column (easy to see)")
for y, v in enumerate(miss_pct.values):
    ax.text(v + 0.1, y, f"{v:.1f}%", va="center", fontsize=8)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "01_missing_pct.png", dpi=150)
plt.show()
print(miss_pct.round(2))

**Insight (simple):** A few behaviour columns (Tenure, hours on app, orders, etc.) are partly blank — about 4–5% each. We will fill them later with medians instead of deleting customers.

**Data-quality notes**

- Seven numeric behavioural columns have missing values (~4–5% each). We will impute in preprocessing (median), not drop rows wholesale.
- No duplicate `CustomerID`s — one row per customer.
- Categorical label overlaps (`Phone`/`Mobile Phone`, `CC`/`Credit Card`, `COD`/`Cash on Delivery`, `Mobile`/`Mobile Phone` in order category) should be harmonised before encoding.

## 4. Target distribution and metric strategy

In [ ]:
churn_rate = df["Churn"].mean()
counts = df["Churn"].value_counts().sort_index()
print(counts)
print(f"Churn rate: {churn_rate:.2%} ({counts.get(1, 0)} / {len(df)})")

import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(["Retained (0)", "Churned (1)"], counts.values, color=["#2a9d8f", "#e76f51"])
ax.set_ylabel("Customers")
ax.set_title("Churn class balance")
for i, v in enumerate(counts.values):
    ax.text(i, v + 40, str(v), ha="center")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "01_churn_balance.png", dpi=150)
plt.show()

In [ ]:
# Visual: churn imbalance as a pie (same story as the bar chart)
fig, ax = plt.subplots(figsize=(5, 5))
labels = ["Retained", "Churned"]
sizes = [counts.get(0, 0), counts.get(1, 0)]
colors = ["#2a9d8f", "#e76f51"]
wedges, texts, autotexts = ax.pie(
    sizes, labels=labels, colors=colors, autopct="%1.1f%%",
    startangle=90, pctdistance=0.75,
    wedgeprops=dict(width=0.45),
)
ax.set_title("Churn share (donut)")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "01_churn_donut.png", dpi=150)
plt.show()

**Insight (simple):** Only about **17%** of customers churned. A model that always says “retained” would look ~83% accurate but catch **zero** churners. That is why we care more about Recall, F1, and PR-AUC than Accuracy.

**Why not Accuracy as the primary metric?**

With ~16.8% churn, a trivial “always retain” classifier is already ~83% accurate. For retention teams:

- **False Negative** (miss a churner) = lost customer + wasted lifetime value.
- **False Positive** (flag a loyal customer) = unnecessary discount / outreach cost.

We therefore prioritise **Recall**, **F1**, and **PR-AUC**, reporting Accuracy only as a secondary sanity check.

## 5. Save interim snapshot

In [ ]:
out_csv = DATA_INTERIM / "ecomm_validated.csv"
df.to_csv(out_csv, index=False)

meta = {
    "n_rows": int(len(df)),
    "n_cols": int(df.shape[1]),
    "churn_rate": float(churn_rate),
    "n_missing_cells": int(df.isna().sum().sum()),
    "duplicate_customer_ids": int(df["CustomerID"].duplicated().sum()),
}
import json
(DATA_INTERIM / "ecomm_validated_meta.json").write_text(json.dumps(meta, indent=2), encoding="utf-8")
print("Saved", out_csv)
print(meta)

## Exit checklist

- Feature dictionary written to `data/interim/feature_dictionary.csv`
- Interim customer table written to `data/interim/ecomm_validated.csv`
- Class imbalance and metric priorities documented
